# G42 — Responder-presence groups on the test split (clip-IV faces for grouping only; no label, no model)

For the descriptive comparison of RoleNet and the baseline by **where the true responder B is visible**, B must be
identified on test MCIS. As in G26 (development), B is the dominant face identity of clip IV matched to an identity
of clips I–III other than the speaker A. This notebook extracts the G8a face features of the **test clip IV** (faces
only, no audio) and writes one group per test MCIS:

* **history only**: B seen in clip I or II, not in III; **current only**: in III, not in I/II; **both**;
* **none**: B found in clip IV but not matched in I–III; **B not found in IV**.

Clip IV is never an input of any model; no label and no prediction is read here. The output CSV is combined offline
with the saved G10 test predictions (`analysis/responder_groups_test.py`).

Inputs: `hi-ef-dataset` (videos of clip IV), `hi-ef-split`, `g8a-features` (clips I–III). GPU recommended.

In [ ]:
!pip install -q insightface onnx
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.22.0"

In [ ]:
# ======== CONFIG ========
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
SPLIT_CSV = "/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv"
OUT_DIR = "/kaggle/working"
SHARD_DIR = f"{OUT_DIR}/g8a_clip4"
SHARD_SIZE = 250

SAMPLE_FPS, MAX_FRAMES = 4, 32
DET_SIZE = (640, 640)
MIN_DET_SCORE, MIN_FACE_PX = 0.5, 24
AUDIO_SR = 16000
WIN_S, HOP_S = 1.5, 0.75           # windowed ECAPA
ENV_HZ = 10                        # RMS energy envelope rate
N_DECODE_THREADS = 4
FER_URL = ("https://github.com/HSE-asavchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/onnx/"
           "enet_b0_8_va_mtl.onnx")
G8A_GLOB = "/kaggle/input/**/shard_*.pkl"          # G8a features of clips I-III (all splits)
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25     # as RoleNet / G26

In [ ]:
import os, glob, pickle, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import torch
import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception as e:
        print("preload_dlls:", e)
PROV = ['CUDAExecutionProvider', 'CPUExecutionProvider']

roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
AUDIO_ROOTS = [os.path.join(r, "audio") for r in roots if os.path.isdir(os.path.join(r, "audio"))]


def find_media(roots_, clip, exts):
    ep, num = clip.split('/')
    for root in roots_:
        for ext in exts:
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


sp = pd.read_csv(SPLIT_CSV, dtype=str, usecols=['sample_id', 'split', 'clip1', 'clip2', 'clip3', 'clip4'])   # no label columns
TEST = sp[sp.split == 'test'].reset_index(drop=True)
clips = sorted(set(TEST.clip4))      # clip IV of the test MCIS only (grouping, never an input)
print(f"MCIS {len(sp)} {sp.split.value_counts().to_dict()} | clips IV (test) {len(clips)} | "
      f"missing video {sum(find_media(VIDEO_ROOTS, c, ('.mp4',)) is None for c in clips)} | "
      f"missing audio {sum(find_media(AUDIO_ROOTS, c, ('.mp3', '.wav')) is None for c in clips)}")

n_miss = sum(find_media(VIDEO_ROOTS, c, ('.mp4', '.avi', '.mkv', '.mov')) is None for c in clips)
assert n_miss <= 0.05 * len(clips), (f"{n_miss}/{len(clips)} clip-IV videos missing: attach hi-ef-dataset (videos); "
                                     "without them every MCIS would fall into 'B not found in IV'")

In [ ]:
from insightface.app import FaceAnalysis
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition', 'landmark_3d_68'], providers=PROV)
app.prepare(ctx_id=0, det_size=DET_SIZE)
print("insightface providers:", {k: m.session.get_providers() for k, m in app.models.items()})

fer_path = f"{OUT_DIR}/enet_b0_8_va_mtl.onnx"
if not os.path.exists(fer_path) or os.path.getsize(fer_path) < 1_000_000:   # re-fetch missing / truncated files
    urllib.request.urlretrieve(FER_URL, fer_path)
assert os.path.getsize(fer_path) > 1_000_000, f"FER model download looks broken ({os.path.getsize(fer_path)} bytes)"
# expose the penultimate (pooled) features as a second output: input of the last Gemm/MatMul
fer_emb_path = f"{OUT_DIR}/enet_b0_8_va_mtl_emb.onnx"
try:
    import onnx
    m = onnx.load(fer_path)
    last = [n for n in m.graph.node if n.op_type in ('Gemm', 'MatMul')][-1]
    m.graph.output.append(onnx.helper.make_tensor_value_info(last.input[0], onnx.TensorProto.FLOAT, None))
    onnx.save(m, fer_emb_path)
    fer = ort.InferenceSession(fer_emb_path, providers=PROV)
except Exception as e:
    print("could not expose the FER embedding, logits only:", repr(e))
    fer = ort.InferenceSession(fer_path, providers=PROV)
FER_IN = fer.get_inputs()[0].name
print("FER providers:", fer.get_providers(), "| outputs", [(o.name, o.shape) for o in fer.get_outputs()])
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)


def fer_batch(crops):
    if not crops:
        return np.zeros((0, 10), np.float32), None
    x = np.stack([(cv2.cvtColor(cv2.resize(c, (224, 224)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255 - MEAN) / STD
                  for c in crops]).transpose(0, 3, 1, 2).astype(np.float32)
    outs = fer.run(None, {FER_IN: x})
    head = outs[0][:, :10] if outs[0].shape[1] >= 10 else np.pad(outs[0], ((0, 0), (0, 10 - outs[0].shape[1])))
    emb = outs[1].reshape(len(x), -1).astype(np.float16) if len(outs) > 1 else None
    return head.astype(np.float32), emb


def read_frames(path):
    # sequential decode, keep ~SAMPLE_FPS frames per second (<= MAX_FRAMES, evenly spread)
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    want = max(1, min(MAX_FRAMES, int(round(n / fps * SAMPLE_FPS)))) if n else MAX_FRAMES
    keep = set(np.linspace(0, max(n - 1, 0), want).astype(int).tolist())
    frames, times, i = [], [], 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if i in keep:
            ok, fr = cap.retrieve()
            if ok:
                frames.append(fr); times.append(i / fps)
        i += 1
        if i > max(keep, default=0):
            break
    cap.release()
    return frames, times, {'n_video_frames': n, 'fps': fps, 'duration': n / fps if fps else 0.0}


def analyse_frames(frames, times):
    dets, crops = [], []
    for fi, fr in enumerate(frames):
        H, W = fr.shape[:2]
        for f in app.get(fr):
            x1, y1, x2, y2 = [float(v) for v in f.bbox]
            if f.det_score < MIN_DET_SCORE or min(x2 - x1, y2 - y1) < MIN_FACE_PX:
                continue
            m = 0.1 * max(x2 - x1, y2 - y1)
            crops.append(fr[int(max(0, y1 - m)):int(min(H, y2 + m)), int(max(0, x1 - m)):int(min(W, x2 + m))])
            lm = getattr(f, 'landmark_3d_68', None)
            pose = getattr(f, 'pose', None)
            d = {'frame': fi, 't': times[fi], 'box': np.array([x1 / W, y1 / H, x2 / W, y2 / H], np.float32),
                 'score': float(f.det_score), 'pose': np.asarray(pose if pose is not None else [0, 0, 0], np.float32),
                 'arc': f.normed_embedding.astype(np.float16)}
            if lm is not None:
                bh = max(y2 - y1, 1.0)
                d['mouth'] = float(np.linalg.norm(lm[66, :2] - lm[62, :2]) / bh)
                d['lm'] = np.stack([(lm[:, 0] - x1) / max(x2 - x1, 1.0), (lm[:, 1] - y1) / bh], 1).astype(np.float16)
            else:
                d['mouth'] = np.nan
            dets.append(d)
    head, emb = fer_batch(crops)
    for k, d in enumerate(dets):
        d['fer'] = head[k]
        if emb is not None:
            d['fer_emb'] = emb[k]
    return dets

In [ ]:
os.makedirs(SHARD_DIR, exist_ok=True)
done = set()
for f in sorted(glob.glob(f"{SHARD_DIR}/c4shard_*.pkl")):
    done |= set(pickle.load(open(f, 'rb')))
todo = [c for c in clips if c not in done]
n_shard = len(glob.glob(f"{SHARD_DIR}/c4shard_*.pkl"))
print(f"already extracted {len(done)} | to do {len(todo)}")


def load(c):
    p = find_media(VIDEO_ROOTS, c, ('.mp4', '.avi', '.mkv', '.mov'))
    return c, (read_frames(p) if p else ([], [], {'n_video_frames': 0, 'fps': 0.0, 'duration': 0.0}))


from collections import deque
from itertools import islice
PREFETCH = 8                        # decoded clips held in memory at most
t0, buf, n_done = time.time(), {}, 0


def flush():
    global n_shard, buf
    if buf:
        pickle.dump(buf, open(f"{SHARD_DIR}/c4shard_{n_shard:03d}.pkl", 'wb'))
        n_shard += 1
        buf = {}
    el = (time.time() - t0) / 60
    print(f"clips {n_done}/{len(todo)} | {el:.1f} min | ETA {el / max(n_done, 1) * (len(todo) - n_done):.0f} min", flush=True)


with ThreadPoolExecutor(N_DECODE_THREADS) as pool:
    it = iter(todo)
    q = deque(pool.submit(load, c) for c in islice(it, PREFETCH))
    while q:
        c, (frames, times, meta) = q.popleft().result()
        nxt = next(it, None)
        if nxt is not None:
            q.append(pool.submit(load, nxt))
        buf[c] = {'meta': {**meta, 'n_sampled': len(frames)}, 'faces': analyse_frames(frames, times),
                  'audio': None}                    # faces only: the grouping needs no audio
        n_done += 1
        if len(buf) >= SHARD_SIZE:
            flush()
flush()
assert n_done == len(todo), (n_done, len(todo))
print("extraction done")

## Groups

In [ ]:
from collections import defaultdict as _dd
from sklearn.cluster import AgglomerativeClustering as _AC

RESP_GROUPS = ['history only', 'current only', 'both', 'none', 'B not found in IV']


def _cluster(E, thr):
    if len(E) < 2:
        return np.zeros(len(E), int)
    return _AC(n_clusters=None, metric='cosine', linkage='average', distance_threshold=1 - thr).fit_predict(E)


def responder_groups(rows, G8, G84, same_cos=0.45, dom_frac=0.25):
    # B = dominant identity of clip IV (>= dom_frac of its sampled frames), matched to a clip I-III identity other
    # than the speaker A (as G26). history = B seen in clip I or II; current = B seen in clip III.
    unit = lambda v: v / (np.linalg.norm(v) + 1e-9)
    out = []
    for row in rows.itertuples():
        cl = [row.clip1, row.clip2, row.clip3]
        items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8.get(c, {}).get('faces', [])))]
        E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else np.zeros((0, 512))
        lab = _cluster(E, same_cos) if len(items) else np.zeros(0, int)
        frames = _dd(set)
        for (k, j), p in zip(items, lab):
            frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
        ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
        A = ids3[0] if ids3 else None
        found, bt = False, None
        f4 = G84.get(row.clip4, {}).get('faces', [])
        if f4:
            E4 = np.stack([d['arc'] for d in f4]).astype(np.float32)
            l4 = _cluster(E4, same_cos)
            fr4 = _dd(set)
            for d, p in zip(f4, l4):
                fr4[p].add(d['frame'])
            dom = max(fr4, key=lambda p: len(fr4[p]))
            if len(fr4[dom]) / max(G84[row.clip4]['meta']['n_sampled'], 1) >= dom_frac:
                found = True
                c4 = unit(E4[l4 == dom].mean(0))
                best, bp = same_cos, None
                for p in set(lab.tolist()):
                    s = float(unit(E[lab == p].mean(0)) @ c4)
                    if s >= best:
                        best, bp = s, p
                bt = bp if bp is not None and bp != A else None
        hist = bt is not None and any(k < 2 and p == bt for (k, _), p in zip(items, lab))
        cur = bt is not None and any(k == 2 and p == bt for (k, _), p in zip(items, lab))
        g = ('B not found in IV' if not found else 'both' if hist and cur else 'history only' if hist
             else 'current only' if cur else 'none')
        out.append({'sample_id': row.sample_id, 'B_found_IV': found, 'B_in_I_II': hist, 'B_in_III': cur, 'group': g})
    return pd.DataFrame(out)

G8 = {}
for f in sorted(glob.glob(G8A_GLOB, recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(TEST[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} test clips I-III missing from g8a-features, e.g. {miss[:3]}"
G84 = {}
for f in sorted(glob.glob(f"{SHARD_DIR}/c4shard_*.pkl")):
    G84.update(pickle.load(open(f, 'rb')))
assert set(clips) <= set(G84), f"{len(set(clips) - set(G84))} clip-IV records missing"
RG = responder_groups(TEST, G8, G84, SAME_PERSON_COS, DOMINANT_MIN_FRAC)
assert len(RG) == len(TEST) == RG.sample_id.nunique()
RG.to_csv(f"{OUT_DIR}/g42_test_responder_groups.csv", index=False)
print(RG.group.value_counts().to_string())
print(f"saved {OUT_DIR}/g42_test_responder_groups.csv  ->  send this file back")